# Cognitive Policies

When making important decisions, humans naturally gather information first, mentally rehearse plans before acting, and assess whether information is reliable. **Cognitive Policies** bring this same multi-round deliberation to your agents — all within a single OTC (observe-think-act) cycle.

There are three cognitive policies:

- **Acquiring** — progressive disclosure of skill instructions and history details on demand
- **Rehearsal** — mental simulation of a planned action before committing
- **Reflection** — assessment of information quality and consistency

In this tutorial, we'll focus on **Acquiring**, which is the built-in, always-active policy and has the most practical use cases: **skill detail disclosure** and **history detail disclosure**. We'll briefly cover Rehearsal and Reflection at the end.

## Initialize

In [1]:
import os

model_name = os.environ.get("MODEL_NAME")
api_key = os.environ.get("API_KEY")
api_base = os.environ.get("BASE_URL")

In [ ]:
from bridgic.llms.openai import OpenAILlm, OpenAIConfiguration

llm = OpenAILlm(
    api_key=api_key,
    api_base=api_base,
    timeout=120,
    configuration=OpenAIConfiguration(
        model=model_name,
        temperature=0.0,
        max_tokens=16384,
    ),
)

In [3]:
from bridgic.core.agentic.tool_specs import FunctionToolSpec


async def get_stock_price(ticker: str) -> str:
    """Fetch stock price data for a ticker symbol"""
    prices = {"AAPL": "$178.50", "GOOGL": "$141.20", "MSFT": "$415.30", "TSLA": "$245.80"}
    return f"{ticker}: {prices.get(ticker, 'Unknown ticker')}"


async def get_financials(company: str) -> str:
    """Get financial metrics for a company"""
    return f"{company} - P/E: 28.5, Revenue Growth: 12%, Debt/Equity: 0.45"


async def analyze_sentiment(ticker: str) -> str:
    """Analyze market sentiment for a stock"""
    return f"Sentiment for {ticker}: 65% bullish, 25% neutral, 10% bearish"


async def generate_recommendation(ticker: str, analysis: str) -> str:
    """Generate an investment recommendation report"""
    return f"Investment Report for {ticker}: {analysis}"


get_stock_price_tool = FunctionToolSpec.from_raw(get_stock_price)
get_financials_tool = FunctionToolSpec.from_raw(get_financials)
analyze_sentiment_tool = FunctionToolSpec.from_raw(analyze_sentiment)
generate_recommendation_tool = FunctionToolSpec.from_raw(generate_recommendation)

## Acquiring — Progressive Disclosure on Demand

Acquiring is the **built-in, always-active** cognitive policy. It enables progressive disclosure through `LayeredExposure` fields — specifically **skills** and **cognitive_history**.

Here's how it works:

1. The LLM sees **summaries** of skills and history steps (not the full content)
2. If the LLM needs more detail, it fills the `details` field in its output with a list of items to reveal
3. The framework **reveals** the requested full content and re-triggers thinking
4. In the next round, the LLM sees the disclosed details and can make an informed decision

Acquiring fires **at most once** per OTC cycle, then closes. It is always active — no flag needed to enable it. It is only disabled when `output_schema` is set on the worker.

**What the LLM sees across rounds:**

- **Round 1 output instructions** include a `details` field: `List[{field: str, index: int}]`
- After the LLM requests details, **Round 2** gets the disclosed details injected into the context and the `details` field is removed from the output instructions

### Acquiring Flow

```
Round 1: LLM sees skill summaries → fills details: [{field: "skills", index: 0}]
         ↓
Framework: reveals skill[0] full content, removes details field from output schema
         ↓
Round 2: LLM sees disclosed details → makes tool call decision
```

## Part 1: Acquiring Skill Details

Skills use `LayeredExposure` — the LLM initially sees only the skill **name** and **description**. If a skill matches its task, the LLM can request the full instructions via the `details` field.

This is perfect for giving agents access to **many procedures** without flooding the context window. The agent only loads the procedure it actually needs.

In [4]:
from bridgic.amphibious import Skill

fundamental_skill = Skill(
    name="fundamental-analysis",
    description="Evaluate a stock's intrinsic value using financial metrics and ratios",
    content=(
        "## Fundamental Analysis Procedure\n\n"
        "Follow these steps to evaluate a stock:\n\n"
        "1. **Get Financial Data**: Retrieve the company's key metrics using get_financials.\n"
        "2. **Evaluate P/E Ratio**:\n"
        "   - P/E < 15: Undervalued (positive signal)\n"
        "   - P/E 15-25: Fairly valued\n"
        "   - P/E > 25: Potentially overvalued (caution)\n"
        "3. **Assess Growth**: Revenue growth > 10% is strong, > 20% is exceptional.\n"
        "4. **Check Debt**: Debt/Equity < 0.5 is healthy, > 1.0 is concerning.\n"
        "5. **Get Current Price**: Use get_stock_price for the latest quote.\n"
        "6. **Rating Logic**:\n"
        "   - BUY: P/E < 20 AND growth > 10% AND debt/equity < 0.7\n"
        "   - SELL: P/E > 35 OR growth < 0% OR debt/equity > 1.5\n"
        "   - HOLD: Everything else\n"
        "7. **Generate Report**: Use generate_recommendation with your analysis summary."
    ),
)

technical_skill = Skill(
    name="technical-analysis",
    description="Analyze stock price patterns, momentum, and technical indicators",
    content=(
        "## Technical Analysis Procedure\n\n"
        "1. Get current and historical price data.\n"
        "2. Calculate RSI (Relative Strength Index):\n"
        "   - RSI > 70: Overbought (potential sell signal)\n"
        "   - RSI < 30: Oversold (potential buy signal)\n"
        "3. Identify support and resistance levels from price history.\n"
        "4. Check momentum: compare recent price to 50-day moving average.\n"
        "5. Generate a technical outlook (bullish/bearish/neutral)."
    ),
)

risk_skill = Skill(
    name="portfolio-risk-assessment",
    description="Evaluate portfolio risk using volatility, sentiment, and diversification metrics",
    content=(
        "## Portfolio Risk Assessment Procedure\n\n"
        "1. Gather sentiment data for each holding using analyze_sentiment.\n"
        "2. Calculate overall portfolio sentiment score.\n"
        "3. Assess concentration risk: single position > 20% is high risk.\n"
        "4. Evaluate volatility based on price fluctuations.\n"
        "5. Assign risk rating: LOW / MEDIUM / HIGH.\n"
        "6. Recommend rebalancing if any position exceeds thresholds."
    ),
)

In [ ]:
from bridgic.amphibious import (
    AmphibiousAutoma, CognitiveContext, CognitiveWorker, think_unit
)


class InvestmentAnalyst(AmphibiousAutoma[CognitiveContext]):
    analyst = think_unit(
        CognitiveWorker.inline(
            "You are an investment analyst. Follow the appropriate skill procedure. "
            "If a skill is available that matches your task, request its details first.",
            verbose_prompt=True,
        ),
        max_attempts=6,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.analyst


agent = InvestmentAnalyst(llm=llm, verbose=True)
result = await agent.arun(
    goal="Perform a fundamental analysis of AAPL and give a BUY/HOLD/SELL rating",
    tools=[get_stock_price_tool, get_financials_tool, analyze_sentiment_tool, generate_recommendation_tool],
    skills=[fundamental_skill, technical_skill, risk_skill],
)
print(result)

[18:03:40.348] [Router] (_amphibious_automa.py:1573) Auto-detecting execution mode
[18:03:40.348] [Router] (_amphibious_automa.py:1579) Detected AGENT mode
[18:03:40.349] [Observe] (_amphibious_automa.py:861) _PromptWorker: None
[18:03:45.256] [Think] (_cognitive_worker.py:332) Message 1 (system, 476 tokens):
You are an investment analyst. Follow the appropriate skill procedure. If a skill is available that matches your task, request its details first.Respond in JSON format.

# Available Tools (with parameters):
• get_stock_price: Fetch stock price data for a ticker symbol
  - ticker (string) [required]
• get_financials: Get financial metrics for a company
  - company (string) [required]
• analyze_sentiment: Analyze market sentiment for a stock
  - ticker (string) [required]
• generate_recommendation: Generate an investment recommendation report
  - ticker (string) [required]
  - analysis (string) [required]

# Available Skills (request details via details: {field: 'skills', index: N})

### What happened?

Look at the verbose output above. The LLM saw **3 skill summaries** (just names and descriptions). It identified that `fundamental-analysis` matched its task and requested its details via the `details` field. The framework then revealed the full procedure, and the LLM followed it step by step — retrieving financials, checking the P/E ratio, evaluating growth, and applying the rating logic.

The other two skills (`technical-analysis` and `portfolio-risk-assessment`) were never loaded into context, keeping token usage efficient.

In [8]:
# Check which skills were acquired (revealed) during execution
for idx, detail in agent.context.skills._revealed.items():
    print(f"[Skill {idx} acquired]")
    print(detail[:200] + "...")

[Skill 0 acquired]
Skill: fundamental-analysis
Description: Evaluate a stock's intrinsic value using financial metrics and ratios

Instructions:
----------------------------------------
## Fundamental Analysis Procedure...


## Part 2: Acquiring History Details

The `CognitiveHistory` also uses `LayeredExposure`, but for the agent's own execution history. It organizes steps into memory layers:

- **Working memory** — most recent steps, shown in full detail
- **Short-term memory** — older steps, shown as summaries only
- **Long-term memory** — oldest steps, compressed into a paragraph

When an agent has many steps, older ones are pushed into short-term or long-term memory. If the LLM needs specifics from an earlier step, it can use the `details` field to request them — the same Acquiring mechanism as with skills.

To see this in action, we'll create a `CognitiveHistory` with a **small working memory** (`working_memory_size=1`) so that even a tool calls are enough to push early steps into short-term memory.

In [ ]:
from bridgic.amphibious import CognitiveHistory


class ThoroughAnalyst(AmphibiousAutoma[CognitiveContext]):
    researcher = think_unit(
        CognitiveWorker.inline(
            "Gather comprehensive data about the stock. "
            "Get price, financials, and sentiment. ",
            verbose_prompt=True,
        ),
        max_attempts=6,
    )
    reporter = think_unit(
        CognitiveWorker.inline(
            "Generate a final investment recommendation based on all gathered data. "
            "If you need details from earlier analysis steps, request them. ",
            verbose_prompt=True,
        ),
        max_attempts=3,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.researcher
        await self.reporter

In [19]:
agent = ThoroughAnalyst(llm=llm, verbose=True)
result = await agent.arun(
    goal="Provide a comprehensive investment analysis of AAPL",
    tools=[get_stock_price_tool, get_financials_tool, analyze_sentiment_tool, generate_recommendation_tool],
    cognitive_history=CognitiveHistory(working_memory_size=1),
)
print(result)

[18:19:49.837] [Router] (_amphibious_automa.py:1573) Auto-detecting execution mode
[18:19:49.837] [Router] (_amphibious_automa.py:1579) Detected AGENT mode
[18:19:49.837] [Observe] (_amphibious_automa.py:861) _PromptWorker: None
[18:19:57.059] [Think] (_cognitive_worker.py:332) Message 1 (system, 363 tokens):
Gather comprehensive data about the stock. Get price, financials, and sentiment. Respond in JSON format.

# Available Tools (with parameters):
• get_stock_price: Fetch stock price data for a ticker symbol
  - ticker (string) [required]
• get_financials: Get financial metrics for a company
  - company (string) [required]
• analyze_sentiment: Analyze market sentiment for a stock
  - ticker (string) [required]
• generate_recommendation: Generate an investment recommendation report
  - ticker (string) [required]
  - analysis (string) [required]

# Context Acquiring
If the context contains progressively disclosed information (e.g. skills, history steps) and you want to inspect the deta

In [20]:
# Inspect the history layers
history = agent.context.cognitive_history
print(f"Total steps: {len(history)}")
print(f"Working memory size: {history.working_memory_size}")
print()
for line in history.summary():
    print(line)

Total steps: 4
Working memory size: 1

[Short-term Memory (0-2), query details via 'details']
  [0] Starting comprehensive investment analysis for AAPL. I need to gather stock price data, financial metrics, and market sentiment to provide a complete analysis. I'll execute multiple data collection tools in parallel to efficiently gather all required information.
  [1] I have gathered the key data for AAPL: stock price ($178.50), financial metrics (P/E: 28.5, Revenue Growth: 12%, Debt/Equity: 0.45), and sentiment score (65). Now I need to generate a comprehensive investment recommendation based on this collected data. I'll compile the analysis and use the generate_recommendation tool to create the final investment report.
  [2] The comprehensive investment analysis for AAPL is complete. All required data has been gathered: stock price ($178.50), financial metrics (P/E: 28.5, Revenue Growth: 12%, Debt/Equity: 0.45), and market sentiment (65/100). The generate_recommendation tool has been 

### What happened?

We configured `CognitiveHistory(working_memory_size=1)`, meaning only the **1 most recent steps** are shown in full detail. The rest are pushed into short-term memory as summaries.

The agent executed in two phases: `researcher` gathered data (price, financials, sentiment), then `reporter` generated the final recommendation. Because working memory is so small, the researcher's early tool calls are already in **short-term memory** by the time the reporter starts.

If the reporter needed specifics from an earlier research step (e.g., the exact P/E ratio from `get_financials`), it could use the `details` field to request the full content — exactly like acquiring skill details.

## Rehearsal — Mental Simulation Before Acting

Rehearsal lets the agent mentally simulate its planned action before committing. This is especially valuable for **high-risk operations** — the agent can catch potential issues before they happen.

With rehearsal enabled, before committing to a tool call, the LLM can fill a `rehearsal` field to simulate what would happen. The framework processes this simulation round and then asks the LLM to make its final decision.

**Use cases:** irreversible actions, financial decisions, operations with significant side effects.

```python
# Enable rehearsal on a think unit
analyst = think_unit(
    CognitiveWorker.inline(
        "Analyze the stock carefully before recommending.",
        enable_rehearsal=True,
    ),
    max_attempts=8,
)
```

## Reflection — Assessing Information Quality

Reflection lets the agent evaluate whether the information it has gathered is sufficient and consistent before acting. The LLM fills a `reflection` field to assess data quality, and the framework injects the assessment into the next round.

**Use cases:** unreliable data sources, contradictory signals, multi-source cross-referencing.

```python
# Enable reflection on a think unit
analyst = think_unit(
    CognitiveWorker.inline(
        "Assess data consistency before recommending.",
        enable_reflection=True,
    ),
    max_attempts=8,
)
```

## Combining Policies

All three policies can be enabled simultaneously. They fire in order: **Acquiring → Rehearsal → Reflection**. Each fires at most once per OTC cycle. This gives your agent the maximum deliberation depth.

```python
analyst = think_unit(
    CognitiveWorker.inline(
        "Perform a comprehensive analysis.",
        enable_rehearsal=True,
        enable_reflection=True,
        # Acquiring is always enabled by default
    ),
    max_attempts=10,
)
```

> **Policy Rules:**
> - Each policy fires **at most once** per observe-think-act cycle, then closes
> - Policies fire in order: **Acquiring → Rehearsal → Reflection**
> - After all active policies have fired, the LLM must commit to a final action
> - Acquiring is built-in and always available; Rehearsal and Reflection must be explicitly enabled
> - Acquiring is disabled when `output_schema` is set on the worker

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we explored how cognitive policies add multi-round deliberation to agents within a single OTC cycle:

- **Acquiring** (built-in, always active): Enables progressive disclosure through `LayeredExposure` fields. The LLM sees summaries first and can request full details on demand.
  - **Skill details**: The agent sees skill names/descriptions, requests the full procedure only when needed — keeping many skills available without flooding context.
  - **History details**: Older execution steps are summarized into memory layers. The agent can request specifics from any earlier step.
- **Rehearsal** (`enable_rehearsal=True`): Lets the LLM mentally simulate its planned action before committing. Ideal for high-risk operations.
- **Reflection** (`enable_reflection=True`): Lets the LLM assess information quality and consistency before acting. Useful when data may be unreliable.
- All policies fire **at most once** per OTC cycle, in the order Acquiring → Rehearsal → Reflection.
- Policies can be combined — enable multiple policies on the same `CognitiveWorker` for maximum deliberation depth.